# IntenCite - LLM pequeno + LoRA + contexto de 3 oraciones + MLflow

Experimento 6 del pool. Se ajusta (fine-tuning) un modelo de lenguaje instruct
pequeno con LoRA para que genere una de las cinco funciones de cita a partir de
la ventana de tres oraciones que rodea la mencion.

- Background
- Gap
- Application
- Improvement
- Comparison

Entrada: columna `context` (ya trae la cita marcada con `<cite>`).
Salida: una etiqueta.

`SMOKE=True` corre una version minima para verificar el circuito en local.
`SMOKE=False` es la corrida real (pensada para Colab con GPU).

## 0. Entorno

En Colab: corre las dos celdas siguientes una vez (clonan el repo, instalan
dependencias y piden subir el CSV). En local no hacen nada.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Colab:", IN_COLAB)

In [ ]:
if IN_COLAB:
    import os

    if not os.path.exists("/content/citation-recommendation-nlp"):
        !git clone -q https://github.com/jtorresor/citation-recommendation-nlp.git /content/citation-recommendation-nlp
    os.chdir("/content/citation-recommendation-nlp")

    !pip install -q "transformers>=5.16" "trl>=1.12" peft accelerate datasets mlflow
    !pip install -q -e . --no-deps

In [ ]:
if IN_COLAB:
    import os
    import shutil

    os.makedirs("data", exist_ok=True)
    if not os.path.exists("data/multicite_3ctx_balanceado.csv"):
        from google.colab import files
        uploaded = files.upload()
        for name in uploaded:
            shutil.move(name, f"data/{name}")

    os.environ["MLFLOW_TRACKING_URI"] = "file:///content/mlruns"
    print(os.listdir("data"))

## 1. Configuracion

In [ ]:
from intencite.config import (MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT)

from pathlib import Path
import json
import os
import tempfile

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from peft import LoraConfig
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

cwd = Path.cwd()
if cwd.name == "experiments":
    ROOT = cwd.parent.parent
elif cwd.name == "notebooks":
    ROOT = cwd.parent
else:
    ROOT = cwd

DATA_PATH = ROOT / "data" / "multicite_3ctx_balanceado.csv"

TEXT_COL = "context"
LABEL_COL = "label_proyecto"
SPLIT_COL = "split_proyecto"
GROUP_COL = "pair_id"

LABELS = ["Application", "Background", "Comparison", "Gap", "Improvement"]

SMOKE = True

if SMOKE:
    BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
    NUM_EPOCHS = 1
    MAX_TRAIN = 48
    MAX_EVAL = 24
else:
    BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    NUM_EPOCHS = 3
    MAX_TRAIN = None
    MAX_EVAL = None

MAX_LEN = 512
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 8

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]

PROMPT_VERSION = "v1"

USE_MLFLOW = True
RUN_NAME = "llm-lora-3ctx-v1"

# En Colab se define os.environ["MLFLOW_TRACKING_URI"] antes de correr.
# En local se usa el valor de src/intencite/config.py.
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", MLFLOW_TRACKING_URI)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("ROOT:", ROOT)
print("DATA_PATH:", DATA_PATH)
print("DEVICE:", DEVICE)
print("BASE_MODEL:", BASE_MODEL)
print("SMOKE:", SMOKE)

## 2. Carga y validacion del dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No encontre {DATA_PATH}. Ejecuta `uv run dvc pull` desde la raiz del repositorio."
    )

df = pd.read_csv(DATA_PATH)

required = {TEXT_COL, LABEL_COL, SPLIT_COL, GROUP_COL}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {missing}")

model_df = df.dropna(subset=[TEXT_COL, LABEL_COL, SPLIT_COL, GROUP_COL]).copy()
model_df[TEXT_COL] = model_df[TEXT_COL].astype(str)
model_df[LABEL_COL] = model_df[LABEL_COL].astype(str)

unknown = set(model_df[LABEL_COL]) - set(LABELS)
if unknown:
    raise ValueError(f"Etiquetas fuera del esquema de cinco clases: {unknown}")

print("Shape:", model_df.shape)
print("\nSplits:")
print(model_df[SPLIT_COL].value_counts())
print("\nClases:")
print(model_df[LABEL_COL].value_counts())

## 3. Particiones oficiales y control de leakage

In [ ]:
train_df = model_df[model_df[SPLIT_COL] == "train"].copy()
val_df = model_df[model_df[SPLIT_COL] == "val"].copy()
test_df = model_df[model_df[SPLIT_COL] == "test"].copy()

train_pairs = set(train_df[GROUP_COL])
val_pairs = set(val_df[GROUP_COL])
test_pairs = set(test_df[GROUP_COL])

assert train_pairs.isdisjoint(val_pairs)
assert train_pairs.isdisjoint(test_pairs)
assert val_pairs.isdisjoint(test_pairs)

def balanced_head(frame, total):
    per_class = total // len(LABELS)
    parts = [
        g.sample(min(len(g), per_class), random_state=SEED)
        for _, g in frame.groupby(LABEL_COL)
    ]
    return pd.concat(parts).sample(frac=1, random_state=SEED)


if MAX_TRAIN is not None:
    train_df = balanced_head(train_df, MAX_TRAIN)
if MAX_EVAL is not None:
    val_df = balanced_head(val_df, MAX_EVAL)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))
print("OK: cero solapamiento de pair_id entre train, val y test.")

## 4. Construccion de los prompts

Cada instancia se convierte en un dialogo de dos turnos: un mensaje de sistema con la
tarea y las cinco definiciones, y un mensaje de usuario con el contexto. La respuesta
esperada (turno del asistente) es la etiqueta.

No se incluyen ejemplos dentro del prompt; esa variante es el experimento 5.

In [ ]:
SYSTEM_PROMPT = (
    "You classify the function of a citation in a scientific paper. "
    "The target reference is wrapped in <cite> </cite>. "
    "Reply with exactly one of these labels and nothing else:\n"
    "Application - the citing work uses an idea, method or tool from the cited work\n"
    "Background - the reference gives context about the domain or the problem\n"
    "Comparison - the citing work points out similarities or differences with the cited work\n"
    "Gap - the reference motivates the work by pointing to an unmet need\n"
    "Improvement - the citing work extends or modifies an idea or method from the cited work"
)


def to_messages(context):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": context},
    ]


def build_dataset(frame):
    rows = [
        {
            "prompt": to_messages(row[TEXT_COL]),
            "completion": [{"role": "assistant", "content": row[LABEL_COL]}],
        }
        for _, row in frame.iterrows()
    ]
    return Dataset.from_list(rows)


train_ds = build_dataset(train_df)
eval_ds = build_dataset(val_df)

print(train_ds)
print(json.dumps(train_ds[0], indent=2, ensure_ascii=False))

## 5. Modelo y tokenizador

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)

print(model.config.model_type, "-", sum(p.numel() for p in model.parameters()) / 1e6, "M parametros")

## 6. Configuracion de LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="CAUSAL_LM",
)

## 7. Entrenamiento

`SFTTrainer` aplica la plantilla de chat, enmascara el prompt (`completion_only_loss`)
y entrena solo los adaptadores LoRA (`peft_config`).

In [ ]:
sft_config = SFTConfig(
    output_dir=tempfile.mkdtemp(prefix="lora_out_"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    max_length=MAX_LEN,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=(DEVICE == "cuda"),
    fp16=(DEVICE == "cuda"),
    logging_steps=5,
    eval_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()
train_result = trainer.train()
print(train_result.metrics)

## 8. Prediccion con probabilidades por clase

Para cada texto se calcula el log-probabilidad medio que asigna el modelo a cada una de
las cinco etiquetas como respuesta, y se normaliza con softmax. La prediccion es la de
mayor probabilidad. Esta salida alimenta la API y el ensemble.

In [ ]:
eval_model = trainer.model
eval_model.eval()
if hasattr(eval_model, "config"):
    eval_model.config.use_cache = True


@torch.no_grad()
def predict_with_probs(frame):
    preds = []
    prob_rows = []
    for context in frame[TEXT_COL]:
        messages = to_messages(context)
        prompt_ids = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=False
        )
        p_len = prompt_ids.shape[1]

        scores = []
        for label in LABELS:
            full = messages + [{"role": "assistant", "content": label}]
            full_ids = tokenizer.apply_chat_template(
                full, return_tensors="pt", return_dict=False
            )
            target = full_ids[0, p_len:]
            logits = eval_model(full_ids.to(DEVICE)).logits[0]
            logprobs = torch.log_softmax(logits.float(), dim=-1)
            step = logprobs[p_len - 1: full_ids.shape[1] - 1, :]
            tok_lp = step.gather(1, target.to(DEVICE).unsqueeze(1)).squeeze(1)
            scores.append(tok_lp.mean().item())

        probs = torch.softmax(torch.tensor(scores), dim=-1).tolist()
        preds.append(LABELS[int(np.argmax(scores))])
        prob_rows.append(dict(zip(LABELS, probs)))

    return preds, pd.DataFrame(prob_rows, index=frame.index)

## 9. Evaluacion en validacion

In [ ]:
def score(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_micro": f1_score(y_true, y_pred, average="micro"),
    }


val_pred, val_probs = predict_with_probs(val_df)
val_metrics = score(val_df[LABEL_COL], val_pred)
val_report = classification_report(
    val_df[LABEL_COL], val_pred, output_dict=True, zero_division=0
)

print(json.dumps(val_metrics, indent=2))
print()
print(classification_report(val_df[LABEL_COL], val_pred, zero_division=0))

## 10. Matriz de confusion de validacion

In [ ]:
cm = confusion_matrix(val_df[LABEL_COL], val_pred, labels=LABELS)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS).plot(
    ax=ax, xticks_rotation=45, values_format="d"
)
ax.set_title("LLM + LoRA (3ctx) - Validation")
plt.tight_layout()
plt.show()

## 11. Registro en MLflow

Para la entrega, `MLFLOW_TRACKING_URI` en `src/intencite/config.py` debe apuntar a la
IP de la EC2. Todas las corridas van al experimento `intencite-model-comparison` con un
`run_name` distinto.

In [ ]:
if USE_MLFLOW:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name=RUN_NAME) as run:
        mlflow.log_params({
            "base_model": BASE_MODEL,
            "method": "LoRA",
            "context": "3ctx",
            "prompt_version": PROMPT_VERSION,
            "decoding": "label-scoring-mean-logprob",
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "lora_targets": ",".join(LORA_TARGETS),
            "learning_rate": LEARNING_RATE,
            "num_epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
            "grad_accum": GRAD_ACCUM,
            "max_len": MAX_LEN,
            "seed": SEED,
            "smoke": SMOKE,
            "train_rows": len(train_df),
            "val_rows": len(val_df),
            "test_rows": len(test_df),
        })

        mlflow.log_metrics({f"val_{k}": float(v) for k, v in val_metrics.items()})

        mlflow.set_tags({
            "project": "IntenCite",
            "stage": "experiment",
            "model_family": "llm_lora",
            "dataset": "multicite_3ctx_proyecto",
        })

        with tempfile.TemporaryDirectory() as tmp:
            tmp = Path(tmp)

            (tmp / "system_prompt.txt").write_text(SYSTEM_PROMPT, encoding="utf-8")
            (tmp / "labels.json").write_text(json.dumps(LABELS), encoding="utf-8")
            (tmp / "classification_report_validation.json").write_text(
                json.dumps(val_report, indent=2), encoding="utf-8"
            )

            cm_path = tmp / "confusion_matrix_validation.png"
            fig, ax = plt.subplots(figsize=(8, 6))
            ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS).plot(
                ax=ax, xticks_rotation=45, values_format="d"
            )
            ax.set_title("LLM + LoRA (3ctx) - Validation")
            plt.tight_layout()
            fig.savefig(cm_path, dpi=160, bbox_inches="tight")
            plt.close(fig)

            pred_out = val_df[[TEXT_COL, LABEL_COL]].copy()
            pred_out["prediction"] = val_pred
            pred_out = pd.concat([pred_out, val_probs.add_prefix("prob_")], axis=1)
            pred_out.to_csv(tmp / "predictions_validation.csv", index=False)

            for name in [
                "system_prompt.txt",
                "labels.json",
                "classification_report_validation.json",
                "confusion_matrix_validation.png",
                "predictions_validation.csv",
            ]:
                mlflow.log_artifact(str(tmp / name), artifact_path="evaluation")

            adapter_dir = tmp / "lora_adapter"
            trainer.model.save_pretrained(str(adapter_dir))
            mlflow.log_artifacts(str(adapter_dir), artifact_path="lora_adapter")

        print("MLflow run ID:", run.info.run_id)
        print("Experiment:", MLFLOW_EXPERIMENT)
        print("Run:", RUN_NAME)
else:
    print("USE_MLFLOW=False: no se registro ninguna corrida.")

## 12. Funcion de inferencia

In [ ]:
def predict_citation_function(text: str) -> dict:
    frame = pd.DataFrame({TEXT_COL: [text]})
    pred, probs = predict_with_probs(frame)
    return {
        "prediction": pred[0],
        "probabilities": {k: float(v) for k, v in probs.iloc[0].items()},
    }


example_text = (
    "Previous work <cite> introduced a similar method for citation intent classification, "
    "which we use as the basis of our experimental setup."
)
predict_citation_function(example_text)

## 13. Test final

**No usar esta seccion para elegir hiperparametros.** El test se evalua una sola vez,
cuando ya se selecciono la mejor configuracion con validacion.

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    test_pred, _ = predict_with_probs(test_df)
    test_metrics = score(test_df[LABEL_COL], test_pred)
    print(json.dumps(test_metrics, indent=2))
    print()
    print(classification_report(test_df[LABEL_COL], test_pred, zero_division=0))
else:
    print("Test reservado. Cambia RUN_FINAL_TEST=True unicamente para la evaluacion final.")